# Day 3 — Pandas: Data Wrangling & Time Series with NIWA Climate Data

## Objective
Load, clean, and analyze tabular climate data using Pandas, covering CSV I/O,
missing-value handling, boolean filtering, groupby aggregation, time series
resampling, and `apply` with lambda — the core data-wrangling toolkit required
before moving to GeoPandas.

## Prerequisites
- **DataFrame / Series**: Pandas' 2D table and 1D column structures
- **CSV I/O**: `pd.read_csv()`, `df.to_csv()`
- **Inspection**: `head()`, `info()`, `describe()`, `dtypes`, `shape`
- **Missing Values**: `isna()`, `dropna()`, `fillna()`, `interpolate()`
- **Filtering & Sorting**: boolean indexing, `sort_values()`, `nlargest()` / `nsmallest()`
- **groupby**: split-apply-combine aggregation, `.agg()`, `.transform()`
- **Time Series**: `pd.to_datetime()`, `resample()`, `rolling()`
- **apply + lambda**: row-wise / column-wise custom transformations

## Dataset
Simulated NIWA climate station records for 5 NZ cities (2020–2024, monthly).
Generated in Cell 2 (no external download required, fully reproducible via `np.random.seed(42)`).

| Field | Type | Description |
|---|---|---|
| `date` | str → datetime | Observation month, stored as the **first day of the month** (`YYYY-MM-01`) by convention |
| `city` | str | Auckland / Wellington / Christchurch / Hamilton / Dunedin |
| `station_id` | str | NIWA station code |
| `temp_c` | float | Monthly mean temperature (°C) |
| `rainfall_mm` | float | Monthly total rainfall (mm) |

- **Volume**: 5 cities × 12 months × 5 years = **300 records**
- **Missing values**: ~3% injected to simulate real station gaps
  (half in `temp_c`, half in `rainfall_mm`)

### Seasonal model
Both variables use the same cosine template, but with **city-specific phase and amplitude**:

```
value(month) = center + amplitude * cos(2π * (month - peak_month) / 12)
```

- **Temperature**: `peak_month = 1` (Southern Hemisphere summer) for all cities.
  ```
  temp = t_mean + t_amp * cos(2π * (month - 1) / 12) + N(0, 0.8)
  ```
- **Rainfall**: `peak_month` **varies by city**, because NZ rainfall seasonality is highly regional:
  | City | Rain peak month | Amplitude | Rationale |
  |---|---|---|---|
  | Auckland | Jul (7) | 0.35 | Winter max — rain-bearing westerlies migrate north in winter |
  | Wellington | Jul (7) | 0.30 | Same North Island westerly regime |
  | Hamilton | Jul (7) | 0.30 | Same North Island westerly regime |
  | Christchurch | Jun (6) | 0.25 | NIWA 1991–2020 normals show June wettest |
  | Dunedin | Jan (1) | 0.15 | Otago rainfall near-uniform; several stations show summer max / winter min |

  ```
  rain_factor = 1.0 + rain_amp * cos(2π * (month - rain_peak) / 12)
  rainfall    = (rain_annual / 12) * rain_factor + N(0, 15), floored at 0
  ```

- **Why multiplicative for rain, additive for temp?** Rainfall cannot be negative
  and city totals differ by 2× (618 mm vs 1240 mm), so a proportional factor keeps
  each city's seasonality scaled to its own magnitude.

⚠️ **Simplification note**: this is a *teaching* dataset. Real NZ climate is driven by
orographic lift, föhn winds, ENSO, and elevation — none of which are modelled here.
A single cosine is a baseline, not a meteorological model. Never present synthetic
data as observation.

## Tasks
- **Task 1** (Cell 3): Load CSV & inspect structure (`head`, `info`, `describe`, `dtypes`)
- **Task 2** (Cell 4): Handle missing values — temperature via per-city linear interpolation (continuous state variable), rainfall via per-city climatology (calendar-month mean, appropriate for event-driven accumulation)
- **Task 3** (Cell 5): Filter & sort — warmest / coldest months, verify SH seasonality
- **Task 4** (Cell 6): `groupby` aggregation — per-city, annual, and city×year summaries
- **Task 5** (Cell 7): Time series — `resample("YE")` yearly means, `rolling(12)` smoothing
- **Task 6** (Cell 8, challenge): `apply` + lambda — add temperature / rainfall category columns

## Expected results (reference)
| Checkpoint | Expected |
|---|---|
| Records | 300 rows |
| Missing after cleaning | 0 |
| Warmest months | All in Jan (Hamilton ~20.1 °C, Auckland ~20.0 °C) |
| Coldest months | All in Jul–Aug (Christchurch ~4.6 °C, Dunedin ~5.8 °C) |
| City mean temp order | Auckland 15.4 > Hamilton 14.0 > Wellington 13.1 > Christchurch 12.1 > Dunedin 11.2 |
| Temp categories | mild 140 / warm 132 / cold 26 / hot 2 |

> Note: `resample("YE")` requires pandas ≥ 2.2. On older versions use `resample("Y")`.

In [ ]:
# Generate simulated NIWA climate data (monthly, 2020-2024, 5 cities)
import numpy as np
import pandas as pd

# city: (annual_mean_temp_c, temp_amp, annual_rainfall_mm, rain_peak_month, rain_amp)
city_profiles = {
    "Auckland": (15.4, 4.0, 1210, 7, 0.35),  # Jul wettest (NIWA)
    "Wellington": (13.2, 4.2, 1240, 7, 0.30),  # Jul wettest
    "Christchurch": (12.1, 5.5, 618, 6, 0.25),  # Jun wettest (NIWA 1991-2020)
    "Hamilton": (14.0, 4.8, 1120, 7, 0.30),  # Jul wettest
    "Dunedin": (11.1, 4.5, 812, 1, 0.15),  # near-uniform, slight summer max
}
station_ids = {
    "Auckland": "A64711",
    "Wellington": "B93451",
    "Christchurch": "C75731",
    "Hamilton": "D87641",
    "Dunedin": "E91961",
}

# rng is a singleton generator
rng = np.random.default_rng(seed=42)  # 42 is the final answer to life, the universe, and the everything ;-)

records = []
for year in range(2020, 2025):
    for month in range(1, 13):
        for city, (t_mean, t_amp, rain_annual, r_peak_month, rain_amp) in city_profiles.items():
            # Temperature: Southern Hemisphere cycle, peak in Jan (month=1)
            seasonal = t_amp * np.cos(2 * np.pi * (month - 1) / 12)
            # N(0, 0.8)
            temp = t_mean + seasonal + rng.normal(0, 0.8)

            # Rainfall: Same cosine Temperature, CITY-SPECIFIC peak month
            rain_factor = 1.0 + rain_amp * np.cos(2 * np.pi * (month - r_peak_month) / 12)
            # N(0, 15)
            rainfall = rain_annual / 12 * rain_factor + rng.normal(0, 15)
            # no rainfall negative
            rainfall = max(rainfall, 0)
            records.append(
                {
                    "date": f"{year}-{month:02d}-01",  # YYYY-MM-DD
                    "city": city,
                    "station_id": station_ids[city],
                    "temp_c": round(temp, 1),
                    "rainfall_mm": round(rainfall, 1),
                }
            )

df_raw = pd.DataFrame(records)
df_len = len(df_raw)
# Inject ~3% missing values to simulate the real station gaps
n_missing = int(df_len * 0.03)
miss_idx = rng.choice(df_len, size=n_missing, replace=False)
df_raw.loc[miss_idx[: n_missing // 2], "temp_c"] = np.nan
df_raw.loc[miss_idx[n_missing // 2 :], "rainfall_mm"] = np.nan

# Save to CSV (simulates downloading from NIWA)
csv_name = "niwa_climate_2020_2024.csv"
df_raw.to_csv(csv_name, index=False)
print(f"Generated {df_len} records -> {csv_name}")
print(f"Missing values injected: {df_raw[['temp_c', 'rainfall_mm']].isna().sum().to_dict()}")
print(df_raw.head())

In [ ]:
# Task1: Load CSV & inspect structure (`head`, `info`, `describe`, `dtypes`)
df = pd.read_csv(csv_name)

print("=== Shape ===")
print(df.shape)

print("\n=== First 8 rows ===")
print(df.head(n=8))  # default is 5

print("\n=== Dtypes & non-null counts ===")
print(df.info())

print("\n=== Statistical summary ===")
print(df.describe().round(1))

print("\n=== Missing values per column ===")
print(df.isna().sum())

print("\n=== Records per city ===")
print(df["city"].value_counts())

In [ ]:
# Task 2: Handle missing values — DIFFERENT strategies for temp vs rainfall
print("Missing before:", df.isna().sum().to_dict())

print("\nRows with missing values:")
print(
    df[df["temp_c"].isna() | df["rainfall_mm"].isna()][["date", "city", "temp_c", "rainfall_mm"]].to_string(index=False)
)

df_clean = df.copy()

# ---------------------------------------------------------------
# Temperature: LINEAR INTERPOLATION (per city)
# Rationale: temperature is a continuous state variable — it changes
# smoothly, so a linear transition between neighbouring months is
# physically reasonable.
# ---------------------------------------------------------------
df_clean["temp_c"] = df_clean.groupby("city")["temp_c"].transform(
    lambda s: s.interpolate(
        method="linear",
        limit_direction="both",  # Edge NaNs (first/last month of a city series) can't be interpolated
    )
)

# ---------------------------------------------------------------
# Rainfall: CLIMATOLOGY (per city, per calendar month)
# Rationale: rainfall is an event-driven accumulation with a right-skewed
# distribution (many small values, occasional extreme storm months).
# Linear interpolation would systematically UNDER-estimate storm months and
# OVER-estimate dry months — it preserves the mean but compresses variance
# and flattens extremes. Standard meteorological practice is to fill with the
# climatological normal (that city's long-term mean for that calendar month),
# which preserves both the seasonal cycle and the variance structure.
# ---------------------------------------------------------------
df_clean["month_num"] = pd.to_datetime(df_clean["date"]).dt.month
climatology = df_clean.groupby(["city", "month_num"])["rainfall_mm"].transform("mean")
df_clean["rainfall_mm"] = df_clean["rainfall_mm"].fillna(climatology)

df_clean = df_clean.drop(df_clean["month_num"])

print("\nMissing after:", df_clean.isna().sum().to_dict())

# Does the filled rainfall still show the same seasonality + spread?
print("\n=== Rainfall stats after climatology fill ===")
print(df_clean["rainfall_mm"].agg(["mean", "std", "min", "max"]).round(1).to_string())


In [ ]:
# Task 3: Filter & sort
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean["year"] = df_clean["date"].dt.year
df_clean["month"] = df_clean["date"].dt.month

print("=== Top 5 warmest months ===")
print(df_clean.nlargest(5, "temp_c")[["date", "city", "temp_c"]].to_string(index=False))

print("\n=== Top5 coldest months ===")
print(df_clean.nsmallest(5, "temp_c")[["date", "city", "temp_c"]].to_string(index=False))

# Verify Southern Hemisphere seasonality: monthly mean temp across all cities
monthly_mean = df_clean.groupby("month")["temp_c"].mean().round(1)
print("\n=== Mean temp by month (expect peak in Jan, trough in Jul) ===")
print(monthly_mean.to_string())

# Boolean filtering: Christchurch summer (Dec-Feb)
chch_summer = df_clean[(df_clean["city"] == "Christchurch") & (df_clean["month"].isin([12, 1, 2]))]
print(f"\nChristchurch summer (DJF) mean temp: {chch_summer['temp_c'].mean():.1f} °C")

# Wettest months
print("\n=== Top5 Wettest months ===")
print(df_clean.nlargest(5, "rainfall_mm")[["date", "city", "rainfall_mm"]].to_string(index=False))


In [ ]:
# Task 4: groupby aggregation — per-city climate summary
city_summary = (
    df_clean.groupby("city")
    .agg(
        mean_temp_c=("temp_c", "mean"),
        min_temp_c=("temp_c", "min"),
        max_temp_c=("temp_c", "max"),
        total_rainfall_mm=("rainfall_mm", "sum"),
        mean_rainfall_mm=("rainfall_mm", "mean"),
        records=("rainfall_mm", "count"),
    )
    .round(1)
)

print("=== Per-city Summary (2020-2024), sorted by mean temp ===")
print(city_summary.sort_values("mean_temp_c", ascending=False).to_string())

annul_summary = (
    df_clean.groupby("year").agg(mean_temp_c=("temp_c", "mean"), total_rainfall_mm=("rainfall_mm", "sum")).round(1)
)

print("\n=== Annul summary (all cities) ===")
print(annul_summary.to_string())

# Multi-level: city x year mean temperature
print("\n=== Mean Temperature: city x year ===")
print(df_clean.groupby(["city", "year"])["temp_c"].mean().unstack("year").round(1).to_string())